# Deep Learning with PyTorch · Overfitting and regularisation

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Little, noisy data

In [ ]:
from torch import nn
from sklearn.datasets import make_moons
Xa, ya = make_moons(40, noise=0.3, random_state=1); Xb, yb = make_moons(300, noise=0.3, random_state=2)
Xtr, ytr = torch.tensor(Xa, dtype=torch.float32), torch.tensor(ya, dtype=torch.float32).unsqueeze(1)
Xva, yva = torch.tensor(Xb, dtype=torch.float32), torch.tensor(yb, dtype=torch.float32).unsqueeze(1)

## 2. Experiment function

In [ ]:
import copy
def deney(p=0.0, wd=0.0, sabir=None, epochs=300, seed=0):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
    opt = torch.optim.AdamW(m.parameters(), lr=0.01, weight_decay=wd); f = nn.BCEWithLogitsLoss()
    best, state, bekle, hist = 1e9, None, 0, []
    for ep in range(epochs):
        m.train(); opt.zero_grad(); f(m(Xtr), ytr).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            tl, vl = f(m(Xtr), ytr).item(), f(m(Xva), yva).item()
        hist.append((tl, vl))
        if vl < best - 1e-4:
            best, state, bekle = vl, copy.deepcopy(m.state_dict()), 0
        else:
            bekle += 1
            if sabir and bekle >= sabir:
                break
    return np.array(hist), best

## 3. Ablation table

In [ ]:
import matplotlib.pyplot as plt
for name, kw in [('baseline', {}), ('dropout 0.3', {'p': 0.3}), ('weight_decay 0.01', {'wd': 0.01}), ('early stopping', {'sabir': 30})]:
    h, best = deney(**kw)
    print(f'{name:22s} val={h[-1, 1]:.3f} best={best:.3f} gap={h[-1, 1] - h[-1, 0]:.3f}')
    plt.plot(h[:, 1], label=name)
plt.legend(); plt.ylabel('val loss'); plt.show()

## Exercises
1. Raise the training samples to 200; what happened to the gap?
2. Repeat with three different seeds and average.
3. Try data augmentation on image data with `transforms.RandomRotation`.